# literature review analysis

**Pipeline position:** stage 4 of 4

this notebook analyzes the final deduplicated dataset from stage 3.

**input:** `02_processed/combined/{date}_final_results.csv`

In [9]:
from pathlib import Path
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
from openai import OpenAI
from dotenv import load_dotenv #to load .env file and retrieve api key
import json #for structured output format in response to llm prompt
from tqdm import tqdm #shows progress bars when looping over entries 
import os
from datetime import date

# Define paths (relative to notebooks/ folder)

output_path = Path("../03_analysis")
output_path.mkdir(exist_ok=True)

In [10]:
# load most recent final results
data_dir = Path("../02_processed/combined")
files = sorted(data_dir.glob("*_final_results.csv"))
latest_file = files[-1]

df = pd.read_csv(latest_file)
print(f"loaded: {latest_file.name}")
print(f"total entries: {len(df)}")
print(f"\nsearch runs:")
print(df['search_run'].value_counts())

loaded: 2026-01-27_final_results.csv
total entries: 502

search runs:
search_run
socialpolicy_covid                                                                 82
globalsouth_countrylist_inequality_poverty_covid_greatrecession                    75
welfarestate_crisis                                                                72
resilience                                                                         67
socialpolicy_crisis                                                                62
welfarestate_covid                                                                 41
manual_wschange                                                                    14
manual_resilience                                                                  14
globalsouth_synon_socialpolicy_inequality_welfarestate_covid_greatrecession_csv    13
ti_resilience_abs_welfarestate                                                     13
welfarestate_greatrecession                                

## 0. generate abstract summaries with llm

In [ ]:
# configurations

##load .env file

load_dotenv()

#retrieve and store key in object, key remains hidden
abssumm_key=os.getenv("openai_abssumm_key") 

#initialize openai client

client=OpenAI(api_key=abssumm_key)

#define model

#model="gpt-5-nano"

#structured output schema

json_schema= {
    "name": "abssumm_schema",
    "schema": {
        "type": "object", #defines type of schema; allowed: string, number, boolean, object...
        "properties": {
            "abssummary": {"type": "string"} #depends on desired nature of output -> abssummaries are strings
        },
        "required": ["abssummary"],
        "additionalProperties": False
    },
    "strict": True #deterministic with regard to output schema
}

In [14]:
# function to summarize abstracts using llm

def summarize_abstracts(df: pd.DataFrame, model:str, abstract_col: str = "abstract") -> pd.DataFrame:
    """
    this function uses an llm to summarize abstracts (column in df) of journal articles (rows in df)

    args:
        df: pandas dataframe containing journal articles with bibliographic info and abstracts as columns.
        model: specifies llm (official name) we want to use, e.g. gpt-4.1-nano 
        abstract_col: specifies name of column we want to draw from 

    returns:
        dict: a dictionary (json) containing each article with an abstract summary
    """
    
    summaries = []
    
    for idx, row in tqdm(df.iterrows(), total=len(df), desc="summarizing abstracts"):
        abstract = row[abstract_col]
        
        # skip if abstract is missing
        if pd.isna(abstract) or str(abstract).strip() == "":
            summaries.append(None)
            continue
        
        try:
            response = client.chat.completions.create(
                model=model,
                messages=[
                    {"role": "system",
                     "content": "you are an expert in summarizing the abstracts of scientific journal articles in a high-quality manner"},
                    {"role": "user",
                     "content": f"summarize this journal article abstract in maximum two sentences in a very concise manner that captures the essential research question, data and results if mentioned:\n\n{abstract}"}
                ],
                response_format={"type": "json_schema", "json_schema": json_schema},
            )
            
            result = json.loads(response.choices[0].message.content)
            summaries.append(result.get("abssummary", None))
            
        except Exception as e:
            print(f"error at index {idx}: {e}")
            summaries.append(None)
    
    df[f"abs_summ_{model}"] = summaries
    return df


In [15]:
#execute function for summaries with gpt-4.1-nano [API call!!] for 2026-01-27_final_results.csv (stored in "df", above)
df=summarize_abstracts(df=df, model="gpt-4.1-nano") 

summarizing abstracts: 100%|██████████| 502/502 [08:08<00:00,  1.03it/s]


In [16]:
# save finres_with_abssumm_df to 03_analysis

df.to_csv(output_path / f"{date.today()}_finres_with_abssumm.csv", index=False)
print(f"saved to {output_path / f'{date.today()}_finres_with_abssumm.csv'}")

saved to ..\03_analysis\2026-01-29_finres_with_abssumm.csv


## 1. explore resilience entries

filter and analyze entries from the resilience search. literature from the manual resilience search run was already studied and integrated into the litrev typst-document and therefore excluded here.

In [17]:
# load enriched finres df with abstract summaries
analysis_dir = Path("../03_analysis")
analysis_files = sorted(analysis_dir.glob("*_finres_with_abssumm.csv"))
latest_analysis_file = analysis_files[-1]

df_enriched = pd.read_csv(latest_analysis_file)
print(f"loaded: {latest_analysis_file.name}")
print(f"total entries: {len(df_enriched)}")

loaded: 2026-01-29_finres_with_abssumm.csv
total entries: 502


In [20]:
# filter manual_resilience entries
resilience_df = df_enriched[df_enriched['search_run'] == 'resilience'].copy()
print(f"resilience entries: {len(resilience_df)}")

# display titles
resilience_df[['title', 'year', 'journal']].head(20)

resilience entries: 67


,title,year,journal
77,Resilience in the face of ecological challenge...,2025,Sustainable Development
87,National eco-social policies in the framework ...,2025,Global Social Policy
110,West European welfare states: The old politics...,2000,European Journal of Political Research
111,Social policy in harsh times. Social security ...,2000,International Journal of Social Welfare
112,Dismantling the social democratic welfare mode...,2001,Scandinavian Political Studies
113,Review essay: Welfare State Challenges and Res...,2001,Acta Sociologica
114,Coping with permanent austerity: Welfare state...,2002,Revue Francaise de Sociologie
115,The silver age of the welfare state: Perspecti...,2002,Journal of Social Policy
116,Risk and Resilience in the Life Course: Implic...,2003,Journal of Youth Studies
122,Political partisanship and welfare state refor...,2004,American Journal of Political Science


Classify publications by their conceptual understanding of resilience (manually, see litrev typstdocument):
- **a) Persistence framing**: Resilience as welfare state persistence against retrenchment, austerity, or neoliberal threats
- **b) Capacities framing**: Resilience as absorptive, adaptive, and transformative capacities (Keck & Sakdapolrak; Pereirinha & Pereira; Daly)
- **c) Neither**: Does not fit either framing

## 2. explore great recession entries

In [21]:
# Filter for Great Recession and related entries
searchruns_filter_gr = [
    "welfarestate_greatrecession",
    "manual_greatrecession",
    "socialpolicy_greatrecession",
    "ti_welfarestate_abs_greatrecession"
]

greatrecession_df = df_enriched[df_enriched['search_run'].isin(search_runs_filter)].copy()
print(f"Great Recession related entries: {len(greatrecession_df)}")
print(f"\nBreakdown by search run:")
print(greatrecession_df['search_run'].value_counts())
greatrecession_df

Great Recession related entries: 29

Breakdown by search run:
search_run
socialpolicy_greatrecession           9
welfarestate_greatrecession           9
manual_greatrecession                 9
ti_welfarestate_abs_greatrecession    2
Name: count, dtype: int64


,title,abstract,eid,authors,journal,issue,volume,keywords,doctype,year,...,pageend,database,search_run,title_normalized,abs_normalized,is_globalsouth,title_abstract,keybert,yake,abs_summ_gpt-4.1-nano
91,Is the Welfare State resisting? Recent develop...,In this paper an attempt was made to analyze s...,WOS:000438917100004,"Kerstenetzky, CL; Guedes, GP",CIENCIA & SAUDE COLETIVA,7,23.0,Welfare State; Organisation for Economic Co-Op...,Article,2018,...,2106,wos,ti_welfarestate_abs_greatrecession,is the welfare state resisting? recent develop...,in this paper an attempt was made to analyze s...,False,is the welfare state resisting? recent develop...,"[('recession', 0.4087), ('2008', 0.2476), ('la...","[('welfare state resisting', 0.007429410766558...",The study analyzes welfare state evolution in ...
92,Welfare State and party political programmes i...,Since the start of the Great Recession in 2007...,WOS:000473527800003,"Ares, C; Losada, A",REVISTA DE ESTUDIOS POLITICOS,184,NaN,Population Ageing; Welfare State; Manifesto Co...,Article,2019,...,101,wos,ti_welfarestate_abs_greatrecession,welfare state and party political programmes i...,since the start of the great recession in 2007...,False,welfare state and party political programmes i...,"[('recession', 0.3248), ('2007', 0.2217), ('de...","[('Great Recession', 0.00402221119755729), ('s...",The study analyzes European political parties'...
132,Resisting austerity measures to social policie...,Since Margaret Thatcher reached power in the U...,2-s2.0-85076526946,"O'Campo, P.; Freiler, A.; Muntañer, C.; Gelorm...",Health Promotion International,6,34.0,Austerity; Employment policies; Fuel poverty; ...,Article,2019,...,1140,scopus,socialpolicy_greatrecession,resisting austerity measures to social policie...,since margaret thatcher reached power in the u...,False,resisting austerity measures to social policie...,"[('thatcher', 0.27), ('neoliberal', 0.2594), (...","[('Margaret Thatcher reached', 0.0009064427839...",The study examines resistance to austerity mea...
133,US Social Policy in the 21st Century: The Diff...,"In Western Europe, political commentators have...",2-s2.0-79959264975,"Daguerre, A.",Social Policy and Administration,4,45.0,American social policy; Economic insecurity; S...,Article,2011,...,407,scopus,socialpolicy_greatrecession,us social policy in the 21st century: the diff...,"in western europe, political commentators have...",False,us social policy in the 21st century: the diff...,"[('obama', 0.2952), ('recession', 0.2462), ('d...","[('President Barack Obama', 0.0009497605608292...",The article analyzes why President Obama's eff...
134,Familistic welfare capitalism in crisis: Socia...,Familistic welfare capitalism is a model of na...,2-s2.0-84890434049,"Papadopoulos, T.; Roumpakis, A.",Journal of International and Comparative Socia...,3,29.0,anti-social policy; familistic welfare capital...,Article,2013,...,224,scopus,socialpolicy_greatrecession,familistic welfare capitalism in crisis: socia...,familistic welfare capitalism is a model of na...,False,familistic welfare capitalism in crisis: socia...,"[('capitalism', 0.3345), ('greece', 0.2939), (...","[('Southern Europe', 0.0036614675598375503), (...",The study investigates familistic welfare capi...
242,European health inequality through the Great R...,This paper investigates the association betwee...,WOS:000431674900010,"van der Wel, KA; Saltkjel, T; Chen, WH; Dahl, ...",SOCIOLOGY OF HEALTH & ILLNESS,4.0,40.0,social determinants of health; social change; ...,Article,2018,...,768,wos,socialpolicy_greatrecession,european health inequality through the great r...,this paper investigates the association betwee...,False,european health inequality through the great r...,"[('recession', 0.3209), ('unemployment', 0.196...","[('great recession', 0.006846512348713536), ('...",The study examines how the Great Recession imp...
280,Weathering the Storm: Social Policy and the Gr...,This review discusses recent challenges to the...,W

In [27]:
# display result
with pd.option_context('display.max_colwidth', None):
    display(greatrecession_df[['title', 'authors', 'year', 'abstract', 'abs_summ_gpt-4.1-nano']])

,title,authors,year,abstract,abs_summ_gpt-4.1-nano
91,Is the Welfare State resisting? Recent developments of the social state in OECD countries,"Kerstenetzky, CL; Guedes, GP",2018,"In this paper an attempt was made to analyze some of the main indicators of the evolution of the welfare state in OECD countries between 1980 and 2016. In particular, an assessment was made to evaluate if the so-called Great Recession starting in 2008 led to a contraction of the social state. The analysis focused on three dimensions: social expenditure, funding, and effectiveness. The conclusion drawn was that the twenty-first century has been a period of expansion, both in terms of social expenditure and the catch-up of the latecomers. In particular, all traditional areas of social policy have expanded in tandem with a slight increase in active social policies. The rise in social expenditure has been financed by increasing taxation not thoroughly alien to progressivity principles. Overall, it has been translated into an increased effort for redistribution. However, inequality and poverty are advancing at a higher rate.","The study analyzes welfare state evolution in OECD countries from 1980-2016, finding overall expansion in social expenditure and policy. Despite increased redistribution efforts financed by progressive taxes, inequality and poverty continue to rise, with the Great Recession not causing a contraction of the social state."
92,Welfare State and party political programmes in Spain: A comparative analysis of parties' social profiles,"Ares, C; Losada, A",2019,"Since the start of the Great Recession in 2007, increased concern has emerged about the amount of social expenditure devoted to the elderly in various European societies. In Spain, this debate has been linked to the irruption of two new political parties Podemos (We can) and Ciudadanos (Citizens). This paper presents a methodological proposal that aims to define, in comparative perspective, parties' social policy profiles. It is based on the annotated manifestos issued by the Manifesto Corpus (Merz et al., 2016) and on the Hirschman's conceptual model. The latter summarises critiques of the Welfare State (WS) in three arguments of perversity, futility, and jeopardy. Data from the Manifesto Project on the 94 general elections held in the EU-15 from 1993 onwards, and covered in the 2017b version of its main dataset (729 observations), are used. Parties' positions on the expansion of welfare spending and emphases on social cuts are examined, both at the meso-level of political parties and the macro-level of national Parliaments. In Spain, Podemos and Ciudadanos have not offered to expand social services for the young generations and, moreover, they have paid scarce attention to the elderly. Neither the Partido Popular (PP Popular Party), nor the Partido Socialista Obrero Espanol (PSOE, Socialist Party), have given greater priority to pensions in comparison to other welfare programmes, such as education or employment. Finally, since 1993, the PP Ciudadanos and Union Progreso y Democracia (UPyD, Union, Progress and Democracy) have been the only parties supportive of cuts to social spending, albeit infrequently. The PP has changed its reasons for proposing cuts to social spending over time. By 2016, these reasons were consistent with the idea of the perversity of the welfare state.","The study analyzes European political parties' social policy profiles from 1993-2016 using manifesto data, revealing that in Spain, parties like Podemos and Ciudadanos show limited support for social spending on the elderly, with mainstream parties like PP and PSOE also not prioritizing pensions and occasionally supporting social cuts aligned with critiques of the welfare state."
132,Resisting austerity measures to social policies: Multiple explanatory case studies,"O'Campo, P.; Freiler, A.; Muntañer, C.; Gelormino, E.; Huegaerts, K.; Puig-Barrachina, V.; Mitchell, C.",2019,"Since Margaret Thatcher reached p

In [45]:
# save greatrecession_df to 03_analysis

greatrecession_df.to_csv(output_path / f"{date.today()}_greatrecession_analysis.csv", index=False)
print(f"saved to {output_path / f'{date.today()}_greatrecession_analysis.csv'}")

saved to ..\03_analysis\2026-01-28_greatrecession_analysis.csv


## 3. explore covid entries

In [ ]:

# Filter for covid entries
searchruns_filter_covid = [
    "welfarestate_covid",
    "manual_covid",
    "socialpolicy_covid"
]

covid_df = df_enriched[df_enriched['search_run'].isin(searchruns_filter_covid)].copy()
print(f"covid related entries: {len(covid_df)}")
print(f"\nbreakdown by search run:")
print(covid_df['search_run'].value_counts())


with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display(covid_df[['authors', 'year', 'title', 'abstract', 'abs_summ_gpt-4.1-nano']])


covid related entries: 131

breakdown by search run:
search_run
socialpolicy_covid    82
welfarestate_covid    41
manual_covid           8
Name: count, dtype: int64


,authors,year,title,abstract,abs_summ_gpt-4.1-nano
1,"Ebbinghaus, B.; Lehner, L.; Naumann, E.",2022,Welfare state support during the COVID-19 pandemic: Change and continuity in public attitudes towards social policies in Germany,"Our analysis asks whether the pandemic situation affects welfare state support in Germany. The pandemic has increased the health and income risks calling for welfare state intervention. While increased needs, more deservingness, and higher state responsibility during such a crisis would suggest augmented support generally and among those at risk, this might be a short-term effect and cost considerations could reverse this trend. We study public attitudes towards four key social policy areas based on the German Internet Panel (GIP). We use three waves prior and further three waves since the pandemic had been declared in March 2020. The analysis shows both continuity in the popularity of social policies, in particular health and pensions, and some short-term increase in support for unemployment and family policies. The results after nearly 2 years suggest rather continuation with some thermostatic short-term boosts in support instead of any long-lasting change. © 2022 The Authors. European Policy Analysis published by Wiley Periodicals LLC on behalf of Policy Studies Organization.","The study examines German public support for social policies before and during the COVID-19 pandemic, finding mostly stable attitudes with temporary boosts in support for unemployment and family policies, and no long-term change in overall welfare support."
2,"Meagher, K.",2022,"Crisis Narratives and the African Paradox: African Informal Economies, COVID-19 and the Decolonization of Social Policy","This article challenges the role of COVID-19 crisis narratives in shaping social policy choices in Africa. The COVID-19 pandemic has focused attention on Africa's vast informal economies, both as a symbol of the continent's intense vulnerability to the ravages of the pandemic, and as a puzzle in the face of the uneven and limited effects of COVID-19 across the continent. Indeed, an examination of statistical and documentary evidence reveals an inverse relationship between COVID-19 fatalities and the size of African informal economies, and a perverse relationship between best-practice COVID social protection responses and levels of COVID-19 mortality. Scrutinizing the evidence behind African COVID-19 crisis narratives raises questions about the ability of donor-led digitized social protection paradigms to address social needs in highly informalized, low-resource environments. This article highlights the role of crisis narratives as an exercise of power geared to remastering, homogenizing and reimagining African informal economies in ways that facilitate particular types of development intervention, sidelining alternative, more socially grounded policy perspectives. Through a closer examination of historical and contemporary realities in Africa's vast and varied informal economies, the article highlights the need to decolonize social policy by privileging local needs and policy perspectives over global policy agendas in the interest of transformative rather than palliative policy responses. © 2022 The Authors. Development and Change published by John Wiley & Sons Ltd on behalf of International Institute of Social Studies.","The article critiques the framing of COVID-19 as a vulnerability in Africa's informal economies, revealing an inverse correlation between fatalities and informal economy size and questioning the efficacy of donor-led digital social protections. It advocates for decolonizing social policy by prioritizing local needs and perspectives over global narratives to foster transformative solutions."
3,"Nayyar, D.; Malhotra, R.",2023,Economic and Social Policies for Human Development,"Economic and social policies of governments could improve or worsen the wellbeing of people, so that their impact on human development could

In [28]:
# save greatrecession_df to 03_analysis

covid_df.to_csv(output_path / f"{date.today()}_covid_analysis.csv", index=False)
print(f"saved to {output_path / f'{date.today()}_covid_analysis.csv'}")

saved to ..\03_analysis\2026-01-29_covid_analysis.csv


## 4. explore 'crisis and welfare state' literature

In [34]:

# Filter for 'crisis and welfare state' entries
searchruns_filter_crisis = [
    "welfarestate_crisis",
    "socialpolicy_crisis",
    "distributionalimpact_crisis"
]

crisisws_df = df_enriched[df_enriched['search_run'].isin(searchruns_filter_crisis)].copy()
print(f"crisis and welfare state related entries: {len(crisisws_df)}")
print(f"\nbreakdown by search run:")
print(crisisws_df['search_run'].value_counts())


with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display(crisisws_df[['authors', 'year', 'title', 'abstract', 'abs_summ_gpt-4.1-nano']])

crisis and welfare state related entries: 139

breakdown by search run:
search_run
welfarestate_crisis            72
socialpolicy_crisis            62
distributionalimpact_crisis     5
Name: count, dtype: int64


,authors,year,title,abstract,abs_summ_gpt-4.1-nano
26,"Berten, J.; Grabo, K.; Kaasch, A.; Martens, K.; Meier, S.; Moore, M.; Panaro, A.V.",2025,"Global social policy ideas in the COVID-19 crisis: Ideational change and continuity in the ILO, the OECD, the WHO, and the World Bank","Global crises constitute challenges for social policy. While social policy is predominantly a national concern, international organisations (IOs) contribute frames of reference for state decisions. In this article, we explore whether the COVID-19 pandemic led to changes in IOs' social policy ideas and recommendations in health care, labour market, and social protection policies due to how IOs perceived the crisis' specific nature, severity, and global scope. We focus on four IOs regarded as key actors in global social policy, namely the ILO, OECD, WHO, and the World Bank. Theoretically, we employ a framework of ideational policy change combining different levels (recommendations - including parameters and instruments - and paradigmatic ideas) with different types of change (layering, conversion, dismantlement, and displacement). We find that IOs have not fundamentally reimagined their pre-pandemic stances during the pandemic. The IOs' perceptions of the crisis do not undermine IOs' ideas and recommendations but highlight their appropriateness. © 2025 The Author(s).","The article examines whether the COVID-19 pandemic led to significant changes in international organizations' social policy recommendations in health, labor, and social protection, focusing on ILO, OECD, WHO, and the World Bank. Findings indicate that these organizations largely maintained their pre-pandemic ideas and advice, seeing their recommendations as appropriate despite the crisis."
27,"Wentzlof, C.A.; Boman Iv, J.H.; Pryor, C.; Hemez, P.",2021,"“Kicking the Can down the Street”: Social Policy, Intimate Partner Violence, and Homicide during the Opioid Crisis","Background: Research has established a strong, positive correlation between homicides and substance use and also between homicides and intimate partner violence (IPV). Additionally, there is a well-known, robust, positive relationship between substance use and IPV. Focusing on the opioid crisis and using county-level panel data, this study investigates the possibility that opioid pill prescription trends, IPV, and homicide are intertwined in a complex, interdependent relationship. Objectives: With an eye toward integrated social policy, this study explores the relationships among and between opioid pill distribution patterns, IPV arrests, and homicide arrests within the United States between 2006 and 2012. Methods: Using county-level panel data from several established federal datasets, a series of longitudinal mixed-effects models regress homicide arrests onto the number of pills within each county, IPV arrests, and other relevant controls. Results: While IPV shares a strong, positive main effect relationship with homicide arrests, opioid pill volume is not significantly related to homicides. However, opioid pill volume within a county significantly attenuates the relationship between IPV and homicide, indicating that pills, IPV, and homicides share a complex, interwoven relationship. Conclusion: While the non-significant main effect of opioid pills suggests that opioids are not consequential for homicide, the highly significant interaction between pills and IPV indicates the opposite. Acknowledging the interwoven link between opioids, IPV, and homicide may be of importance when attempting to develop individual programs aimed at addressing these social problems. Well intentioned, unidimensional policies aimed at reducing opioids or IPV may be unknowingly impacting other social problems. © 2021 Taylor & Francis Group, LLC.","This study analyzes county-level data from 2006-2012 to explore the intertwined relationships among opioid pill distribution, intimate partner violence (IPV), and homicides in the US. Results show IPV's s

In [37]:
#this section divides the dataframe crisisws_dw into literature that mention covid or financial crisis or neither or both in their title

# Define search patterns for title_normalized (already lowercase)
covid_terms = [
    "covid", "covid-19", "coronavirus", "sars-cov2", 
    "global pandemic", "corona", "pandemic"
]

financial_crisis_terms = [
    "great recession", "subprime mortgage crisis", "subprime crisis",
    "sovereign debt crisis", "housing bubble", "financial crisis"
]

# Create regex patterns
covid_pattern = "|".join(covid_terms)
financial_crisis_pattern = "|".join(financial_crisis_terms)

# Create boolean masks
has_covid = crisisws_df['title_normalized'].str.contains(covid_pattern, na=False, regex=True)
has_financial_crisis = crisisws_df['title_normalized'].str.contains(financial_crisis_pattern, na=False, regex=True)

# Handle special case: "wall street crash" AND "2008"
has_wall_street_2008 = (
    crisisws_df['title_normalized'].str.contains("wall street crash", na=False) & 
    crisisws_df['title_normalized'].str.contains("2008", na=False)
)
has_financial_crisis = has_financial_crisis | has_wall_street_2008

# Create three dataframes
subs_crisis_covid_df = crisisws_df[has_covid & ~has_financial_crisis].copy()
subs_crisis_greatrec_df = crisisws_df[has_financial_crisis & ~has_covid].copy()
subs_crisis_both_df = crisisws_df[has_covid & has_financial_crisis].copy()
subs_crisis_neither_df = crisisws_df[~has_covid & ~has_financial_crisis].copy()

# Print summary
print(f"COVID-related titles: {len(subs_crisis_covid_df)}")
print(f"Financial crisis titles: {len(subs_crisis_greatrec_df)}")
print(f"Both crises mentioned: {len(subs_crisis_both_df)}")
print(f"Neither crisis: {len(subs_crisis_neither_df)}")
print(f"Total: {len(subs_crisis_covid_df) + len(subs_crisis_greatrec_df) + len(subs_crisis_both_df) + len(subs_crisis_neither_df)}")


COVID-related titles: 9
Financial crisis titles: 13
Both crises mentioned: 1
Neither crisis: 116
Total: 139


In [39]:
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display(subs_crisis_greatrec_df[['authors', 'year', 'title', 'abstract', 'abs_summ_gpt-4.1-nano']])

,authors,year,title,abstract,abs_summ_gpt-4.1-nano
29,"Vetterlein, A.",2015,Paradigm maintenance: The IMF and social policies after the financial crisis,NaN,NaN
33,"Min, S.",2016,The Chinese orientation of social policies of new socialized old-age insurance system in rural areas in the post financial crisis era,"As one of the important social policies, the new socialized old-age insurance system in rural areas has played a vital role in the construction of Chinese social security system in the post financial crisis era. Manifesting national responsibilities, it has embodied the core concept of social policies, and had a great significance for boosting domestic demand. Therefore, this thesis, on the basis of this new system, can continually perfect Chinese pension security system and promote the sustainable development of the social security course. © 2016, Canadian Center of Science and Education. All rights reserved.","The study examines China's rural old-age insurance system, emphasizing its role in social security and domestic demand; it aims to improve and sustain China's pension security framework for social policy development."
54,"Saxonberg, S; Sirovátka, T",2014,From a Garbage Can to a Compost Model of Decision-Making? Social Policy Reform and the Czech Government's Reaction to the International Financial Crisis,"This article argues that the financial crisis did not have a strong impact on Czech social policy. In contrast to the garbage-can model, in which policymakers wait for a 'window of opportunity' to implement radical reforms, the Czech centre-right parties instead used the crisis as a means of continuing their market-liberal reforms that they had initiated before the crisis had even begun. Since they had extremely little public support for their reforms, they tried to bring about gradual change. Thus, instead of pulling radical reforms out of the garbage-can, Czech right-wing politicians have tended to take out smaller biodegradable goods that can decay and compost into fertile ground in order to sow the seeds of gradual change.","The article finds that the Czech social policy was minimally affected by the financial crisis, as right-wing parties utilized the crisis to further pre-existing market-liberal reforms through gradual, manageable changes rather than radical revolutions, despite limited public support."
129,"Ulriksen, M.S.",2011,Social policy development and global financial crisis in the open economies of Botswana and Mauritius,"The manner in which open economies in a globalized world shape social policy development is highly disputed, as is the impact of the current financial crisis on social policy. One argument is that globalization and economic austerity force social policy dismantling. Alternatively, it is proposed that open economies - facing greater volatility, especially during crisis - push for greater social protection. Using the examples of Botswana and Mauritius, two open middle-income countries, this article suggests that, in fact, both arguments may be correct. The impact of globalization and economic crisis depends on the character of the welfare system already in place and the organized interests underpinning it. In Botswana a main social policy thrust is to increase efficiency in spending, whereas issues of job security and compensation are more prevalent in Mauritius. The findings imply that divergence across welfare systems persists and that, even in crisis, countries often use tried social policy solutions. © The Author(s) 2011.","The article examines how globalization and economic crises influence social policy in Botswana and Mauritius, revealing that effects depend on existing welfare systems, with Botswana focusing on efficiency and Mauritius on job security; both countries tend to rely on familiar social policies despite economic challenges."
177,"Schwartz, H.",2012,"Housing, the welfare state, and the global financial crisis: What is the connection?","Analyses of the global fin

## 5. explore global south entries

In [5]:
# Filter for Global South entries
searchruns_filter_globalsouth = [
    "globalsouth_countrylist_inequality_poverty_covid_greatrecession",
    "globalsouth_synon_socialpolicy_inequality_welfarestate_covid_greatrecession_csv",
    "globalsouth_countrylist_socialpolicy_covid_greatrecession",
    "manual_globalsouth"
]

globalsouth_df = df_enriched[df_enriched['search_run'].isin(searchruns_filter_globalsouth)].copy()
print(f"Global South related entries: {len(globalsouth_df)}")
print(f"\nbreakdown by search run:")
print(globalsouth_df['search_run'].value_counts())



with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display(globalsouth_df[['authors', 'year', 'title', 'abstract', 'abs_summ_gpt-4.1-nano']])


Global South related entries: 95

breakdown by search run:
search_run
globalsouth_countrylist_inequality_poverty_covid_greatrecession                    75
globalsouth_synon_socialpolicy_inequality_welfarestate_covid_greatrecession_csv    13
globalsouth_countrylist_socialpolicy_covid_greatrecession                           5
manual_globalsouth                                                                  2
Name: count, dtype: int64


,authors,year,title,abstract,abs_summ_gpt-4.1-nano
0,"Faria, L; Alvarez, REC; Santos, LAD",2023,Socioeconomic inequality in Latin America and the Caribbean: the post-pandemic future for the training of health professionals,"Inequality is a global, structural problem that is particularly marked in the world's poorest countries. The covid-19 pandemic exacerbated this historic problem in Latin America and the Caribbean and deepened uncertainties in relation to basic human needs. This study presents an overview of the subject on the basis of official reports from international agencies (PAHO, WHO, ECLAC) between 2019 and 2022 and discusses some paths for the training of health professionals in Brazil. It also investigates how health practices could be changed to ensure greater social protection for vulnerable populations, based on the proposals of Paulo Freire and Edgar Morin, which highlight current social and health problems.","The study reviews official reports from 2019-2022 on inequality and health in Latin America, focusing on the COVID-19 pandemic's impact and exploring ways to enhance social protection through educational and ethical practices inspired by Freire and Morin to address vulnerabilities."
206,"Dantas, MNP; Silva, MDD; Barbosa, IR",2022,Reflections on the COVID-19 mortality among the Black population and racial inequality in Brazil,"This literature critical analysis reflects on the social, political, and historical background responsible for racial discrepancies in hospital mortality by COVID-19 among the Brazilian population. During the pandemic, the COVID-19 mortality among the Black population gained notoriety. Rather than an isolated fact, this finding has historical roots dating back to Brazil???s foundation and draws on structural racism, which reveals degrading living and health conditions experienced by the Black population before the pandemic. This situation of vulnerability affecting the Black population is a recurring scenario that is treated with the neglect inherent to structural racism. COVID-19 mortality portrays one way in which racism impacts and reproduces itself in the life and death of Black people.","This analysis examines how structural racism contributes to racial disparities in COVID-19 hospital mortality among Brazilians, highlighting historical and social factors leading to higher mortality rates among Black populations. It underscores that COVID-19 mortality is a manifestation of ongoing racial inequality rooted in Brazil's history of systemic racism."
207,"Wu, XF; Sun, ZW; Cheng, JL",2022,Who suffers from COVID-19 shock? Labour market risk and wage inequality in China,"Through an online survey from 2020, this study aims to investigate how labour market status and occupational factors shape the risk of wage drops resultant of the COVID-19 outbreak. Quantitative evidence shows the risk is amplified for lower-educated labourers working in administrative positions, and affiliated to the traditional service sector and foreign invested enterprises (FIE) and they are more likely to experience earning reduction. COVID-19 in China witnesses a significantly downward trend of labours` wage and general wage inequality has slightly eased. This study calls for policy shift in direct financial subsidies to workers to mitigate labours' risk.","This study investigates how labour market status and occupational factors affected wage drops due to COVID-19, with findings indicating higher risk for lower-educated workers in administrative, service, and foreign-invested sectors. Results show a general decline in wages with slight easing of wage inequality, suggesting policy needs for targeted financial support to workers."
208,"Ma, GM; Zhang, JY; Hong, L",2023,Learning From Home: Widening Rural-Urban Educational Inequality and High School Students' Self-Control in China During the COVID-19 Pandemic and School Closure,"Worldwide school closures and remote learning have been implemented during the COVID-19 pa